In [ ]:
import pandas as pd
import numpy as np
import re
import ipaddress
import joblib

from urllib.parse import urlsplit

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import matplotlib.pyplot as plt
import seaborn as sns

In [1]:
import pandas as pd
import zipfile
import urllib.request

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

urllib.request.urlretrieve(url, "sms_dataset.zip")

with zipfile.ZipFile("sms_dataset.zip", "r") as zip_ref:
    zip_ref.extractall("sms_data")

print("Dataset downloaded successfully!")

Dataset downloaded successfully!


In [2]:
df = pd.read_csv(
    "sms_data/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)

print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nLabel Distribution:")
print(df["label"].value_counts())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Dataset Shape:
(5572, 2)

Label Distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


In [3]:
import re

def clean_text(text):

    text = text.lower()

    text = re.sub(r'http\S+|www\S+', ' URL ', text)

    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text


df["message"] = df["message"].apply(clean_text)

print(df.head())

  label                                            message
0   ham  go until jurong point crazy available only in ...
1   ham                            ok lar joking wif u oni
2  spam  free entry in 2 a wkly comp to win fa cup fina...
3   ham        u dun say so early hor u c already then say
4   ham  nah i don t think he goes to usf he lives arou...


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

X = df["message"]

y = df["label"]

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_vectorized = vectorizer.fit_transform(X)

print("Feature matrix:", X_vectorized.shape)

Feature matrix: (5572, 5000)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 4457
Testing samples: 1115


In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

print("SMS Detection Model Trained Successfully!")

SMS Detection Model Trained Successfully!


In [7]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

y_pred = model.predict(X_test)

print("Accuracy:",
      accuracy_score(y_test, y_pred) * 100)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 98.02690582959642

Classification Report:
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.92      0.93      0.93       149

    accuracy                           0.98      1115
   macro avg       0.96      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115


Confusion Matrix:
[[954  12]
 [ 10 139]]


In [8]:
def detect_sms(message):

    cleaned_message = clean_text(message)

    message_vector = vectorizer.transform(
        [cleaned_message]
    )

    prediction = model.predict(message_vector)[0]

    probabilities = model.predict_proba(
        message_vector
    )[0]

    if prediction == "spam":

        print("WARNING: SPAM / SCAM SMS")

    else:

        print("SMS classified as NORMAL")

    print("Prediction:", prediction)

    print("Model scores:", probabilities)


# Test SMS
sms = "Congratulations! You won a free prize. Click this link now!"

detect_sms(sms)

Prediction: spam
Model scores: [0.06133956 0.93866044]


In [ ]:
while True:

    sms = input("\nEnter SMS (type exit to stop): ")

    if sms.lower() == "exit":
        break

    detect_sms(sms)

Prediction: spam
Model scores: [0.1848493 0.8151507]
SMS classified as NORMAL
Prediction: ham
Model scores: [0.66073977 0.33926023]


In [ ]:
import joblib

joblib.dump(model, "sms_model.pkl")

joblib.dump(vectorizer, "sms_vectorizer.pkl")

print("Model saved successfully!")

In [ ]:
from google.colab import files

files.download("sms_model.pkl")
files.download("sms_vectorizer.pkl")